In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor
from functools import partial

import pandas as pd
from tqdm import tqdm
from pathlib import Path

import sys

p = Path.cwd().resolve()
while p != p.parent and not (p / "causal.py").exists():
    p = p.parent

BASE = p
sys.path.insert(0, str(BASE / "extraction" / "PARKCELEB"))

from features.interpretable_features.collectors import (
    get_pause_features, get_syllable_features, get_intensity_features
)

AUDIO_DIR = BASE / "data" / "neurovoz" / "audios"
OUT_DIR = BASE / "data" / "neurovoz" / "extracted_features"
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
folder = AUDIO_DIR
n_files = sum(p.is_file() for p in folder.iterdir())
print(n_files)


In [ ]:
def read_audio_files_neurovoz(input_dir):
    """
    Reads all .wav audio files from a directory tree.

    Args:
        input_dir (str): Root directory to search for audio files.

    Returns:
        list: A list of file paths for the audio files.
    """
    all_audios = []
    for path, subdirs, files in os.walk(input_dir):
        for name in files:
            if name.endswith(".wav"):
                file_path = os.path.join(path, name)

                if os.path.getsize(file_path) >= 32_768:
                    all_audios.append(file_path)
                else:
                    print(file_path, os.path.getsize(file_path))

    return all_audios


In [ ]:
files = read_audio_files_neurovoz(str(AUDIO_DIR))


In [ ]:
pause_features = []
method = partial(get_pause_features, max_duration=None)
with ProcessPoolExecutor(max_workers=2) as executor:
    with tqdm(total=len(files)) as pbar:
        for feat in executor.map(method, files):
            pbar.update(1)
            pause_features.append(feat)


In [ ]:
syllable_features = []
method = partial(get_syllable_features, max_duration=None)
with ProcessPoolExecutor(max_workers=2) as executor:
    with tqdm(total=len(files)) as pbar:
        for feat in executor.map(method, files):
            pbar.update(1)
            syllable_features.append(feat)


In [ ]:
intensity_features = []
method = partial(get_intensity_features, max_duration=None)
with ProcessPoolExecutor(max_workers=2) as executor:
    with tqdm(total=len(files)) as pbar:
        for feat in executor.map(method, files):
            pbar.update(1)
            intensity_features.append(feat)


In [ ]:
pd.DataFrame(pause_features).to_csv(OUT_DIR / "pause_features_neurovoz.csv")
pd.DataFrame(syllable_features).to_csv(OUT_DIR / "syllable_features_neurovoz.csv")
pd.DataFrame(intensity_features).to_csv(OUT_DIR / "intensity_features_neurovoz.csv")


In [ ]:
base = OUT_DIR
base.resolve()


In [ ]:

intensity_path = base / "intensity_features_neurovoz.csv"
pause_path = base / "pause_features_neurovoz.csv"
syllable_path = base / "syllable_features_neurovoz.csv"
if any(not p.exists() for p in [intensity_path, pause_path, syllable_path]):
    raise FileNotFoundError("Feature file not found! Have you picked the correct extract length?")


intensity = pd.read_csv(intensity_path, index_col=0)
pause = pd.read_csv(pause_path, index_col=0)
syllable = pd.read_csv(syllable_path, index_col=0)
features_full = pd.concat(objs=[
    intensity.add_prefix("int_"), pause.add_prefix("pause_"), syllable.add_prefix("syl_")
], axis="columns")


index = []
for i, l in features_full.iterrows():
    patient = l["int_tot_name"][-8:-4]
    test = l["int_tot_name"].split("/")[-1][3:-9]
    index.append((patient, test))

    assert patient in l["pause_tot_name"]
    assert test in l["pause_tot_name"]
    assert patient in l["syl_tot_name"]
    assert test in l["syl_tot_name"]


features = features_full.drop(columns=["int_tot_name", "pause_tot_name", "syl_tot_name"])
features.index = pd.MultiIndex.from_tuples(index, names=["patient", "test"])


## role in dissertation

this notebook is the first stage of the dissertation's pipeline. it
walks every wav under `data/neurovoz/audios/`, runs the upstream
parkceleb interpretable-feature collectors (pause, syllable,
intensity) over the corpus in parallel, and writes the three
per-clip csvs to `data/neurovoz/extracted_features/`. those csvs are
exactly the inputs `old/causal.py:load_nv` expects when it builds
the 66-dim acoustic feature matrix used by every downstream arm:

* the neurovoz within-corpus deconfounding arms in `main.py`
  (`pipeline_variants_nv.csv`, `pipeline_samecorpus_nv.csv`),
* the ugas age+sex back-door transfer in
  `tests/transport_u_duration.py` (`pipeline_transfer.csv`),
* the stage-2 raw-feature sweep in `tests/transport_u_age.py`,
* and the wav2vec2-vs-hand-crafted comparison in
  `tests/ssl_transfer_compare.py`.

without the csvs produced here, `old/causal.load_nv` falls back to
an empty feature matrix and the entire neurovoz side of the
pipeline is silent. the same pause / syllable / intensity scheme is
also applied to parkceleb in `extraction/parkceleb/`; keeping the
two feature sets symmetric is what makes the cross-corpus transfer
to parkceleb disease-duration intervals possible.